# Phase 5 — Tests & Config
**Project:** MoodSyncAI — Multi-Modal Sentiment & Emotion Analyser  
**Module:** Data Analytics-3

This notebook is a *run-book* companion to the four `.py` test files in `tests/` and the three YAML configs in `training/configs/`.  
It lets you execute every test suite and validate every config in one place.

| File | Purpose |
|------|---------|
| `tests/test_visual_model.py` | Face detector · CNN emotion model · Grad-CAM · Polarity mapping |
| `tests/test_text_model.py` | Preprocessing · RoBERTa sentiment · Attention weights |
| `tests/test_fusion.py` | Mismatch detector · Fusion layer · Polarity bridge |
| `tests/test_generator.py` | GPT-2 / LLM summary generator · Prompt builder |
| `training/configs/cnn_config.yaml` | ResNet50V2 training hyper-parameters |
| `training/configs/text_config.yaml` | RoBERTa inference / fine-tuning settings |
| `training/configs/fusion_config.yaml` | Fusion weights · mismatch thresholds |


## 0  Environment

In [ ]:
import sys, subprocess, pathlib, warnings
warnings.filterwarnings("ignore")

print(f"Python  {sys.version.split()[0]}")
print(f"CWD     {pathlib.Path.cwd()}")

# Make the repo root importable (adjust depth if running from notebooks/)
repo_root = pathlib.Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print(f"sys.path[0]  {sys.path[0]}")


## 1  Run All Test Suites

Each cell runs one `unittest` module via `subprocess` so output is captured cleanly.  
**Expected:** `OK` on all suites once the corresponding `models/` files are implemented.


In [ ]:
import subprocess, sys

def run_tests(module: str):
    result = subprocess.run(
        [sys.executable, "-m", "pytest", f"tests/{module}", "-v", "--tb=short"],
        capture_output=True, text=True
    )
    print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
    if result.returncode != 0:
        print(result.stderr[-1500:])
    return result.returncode


### 1a  Visual Model Tests

In [ ]:
run_tests('test_visual_model.py')

### 1b  Text Model Tests

In [ ]:
run_tests('test_text_model.py')

### 1c  Fusion Tests

In [ ]:
run_tests('test_fusion.py')

### 1d  Generator Tests

In [ ]:
run_tests('test_generator.py')

### 1e  Full Suite (one shot)

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/", "-v", "--tb=short", "--no-header"],
    capture_output=True, text=True
)
print(result.stdout[-5000:] if len(result.stdout) > 5000 else result.stdout)
print("Return code:", result.returncode)


## 2  Validate YAML Configs

Load each config and print a structured summary so you can verify the values
match the hyper-parameters chosen during training.


In [ ]:
import yaml, json, pathlib

CONFIG_DIR = pathlib.Path("training/configs")

def load_and_print(filename: str):
    path = CONFIG_DIR / filename
    with open(path) as f:
        cfg = yaml.safe_load(f)
    print(f"\n{'='*60}")
    print(f"  {filename}")
    print('='*60)
    print(json.dumps(cfg, indent=2, default=str))
    return cfg

cnn_cfg    = load_and_print("cnn_config.yaml")
text_cfg   = load_and_print("text_config.yaml")
fusion_cfg = load_and_print("fusion_config.yaml")


## 3  Config Assertion Checks

In [ ]:
# ── CNN config ───────────────────────────────────────────────────────────────
assert cnn_cfg["model"]["num_classes"] == 7,                "CNN must have 7 output classes"
assert set(cnn_cfg["model"]["classes"]) == {
    "Angry","Disgust","Fear","Happy","Neutral","Sad","Surprise"
},                                                          "All FER-2013 classes required"
assert cnn_cfg["model"]["input_shape"] == [48, 48, 3],      "Standard FER-2013 input size"
assert cnn_cfg["training"]["batch_size"] >= 16,             "Batch size too small"
assert 0.0 < cnn_cfg["training"]["learning_rate"] < 1.0,   "Unreasonable learning rate"

# ── Text config ───────────────────────────────────────────────────────────────
assert text_cfg["model"]["num_labels"] == 3,               "RoBERTa must have 3 labels"
assert "negative" in text_cfg["model"]["labels"]
assert "neutral"  in text_cfg["model"]["labels"]
assert "positive" in text_cfg["model"]["labels"]
assert text_cfg["model"]["visual_to_polarity"]["Happy"]   == "positive"
assert text_cfg["model"]["visual_to_polarity"]["Sad"]     == "negative"
assert text_cfg["model"]["visual_to_polarity"]["Neutral"] == "neutral"
assert 0.0 < text_cfg["mismatch"]["conf_threshold"] < 1.0, "Threshold out of range"

# ── Fusion config ─────────────────────────────────────────────────────────────
vw = fusion_cfg["fusion"]["weighted_average"]["visual_weight"]
tw = fusion_cfg["fusion"]["weighted_average"]["text_weight"]
assert abs(vw + tw - 1.0) < 1e-6,                         "Fusion weights must sum to 1"
assert "HARD_MISMATCH" in fusion_cfg["mismatch"]["severity_levels"]
assert "SOFT_MISMATCH" in fusion_cfg["mismatch"]["severity_levels"]
assert "MATCH"         in fusion_cfg["mismatch"]["severity_levels"]

print("\n✅  All config assertions passed.")


## 4  Polarity Bridge Coverage Check

In [ ]:
visual_classes  = ["Angry","Disgust","Fear","Happy","Neutral","Sad","Surprise"]
polarity_map    = text_cfg["model"]["visual_to_polarity"]

missing  = [c for c in visual_classes if c not in polarity_map]
invalid  = [c for c, p in polarity_map.items()
            if p not in ("positive","neutral","negative")]

print(f"Visual classes  : {visual_classes}")
print(f"Polarity map    : {polarity_map}")
print(f"Missing entries : {missing  or 'none'}")
print(f"Invalid values  : {invalid  or 'none'}")

assert not missing, f"Classes not covered: {missing}"
assert not invalid, f"Invalid polarity values: {invalid}"
print("\n✅  Polarity bridge is complete and valid.")


## 5  Mismatch Logic Matrix (Smoke-Test)

In [ ]:
test_pairs = [
    # visual_pred  vis_conf  text_pred  txt_conf  expected_severity
    ("Happy",   0.80, "positive", 0.81, "MATCH"),
    ("Sad",     0.68, "positive", 0.81, "HARD_MISMATCH"),   # assignment brief
    ("Fear",    0.55, "neutral",  0.60, "HARD_MISMATCH"),
    ("Neutral", 0.72, "neutral",  0.75, "MATCH"),
    ("Sad",     0.35, "positive", 0.82, "SOFT_MISMATCH"),   # low visual conf
    ("Happy",   0.80, "negative", 0.82, "HARD_MISMATCH"),
    ("Angry",   0.40, "positive", 0.44, "SOFT_MISMATCH"),   # both uncertain
    ("Disgust", 0.90, "negative", 0.85, "MATCH"),
]

POLARITY = {
    "Happy":"positive","Surprise":"positive","Neutral":"neutral",
    "Angry":"negative","Disgust":"negative","Fear":"negative","Sad":"negative",
}
THRESHOLD = fusion_cfg["mismatch"]["conf_threshold"]

def detect_mismatch_inline(vp, vc, tp, tc, thresh):
    vp_pol = POLARITY.get(vp, "neutral")
    match  = (vp_pol == tp)
    both_c = (vc >= thresh) and (tc >= thresh)
    if match:
        return "MATCH"
    return "HARD_MISMATCH" if both_c else "SOFT_MISMATCH"

print(f"{'Visual':10} {'VConf':6} {'Text':10} {'TConf':6} {'Expected':16} {'Got':16} {'OK':3}")
print("-"*70)
all_pass = True
for vp, vc, tp, tc, expected in test_pairs:
    got = detect_mismatch_inline(vp, vc, tp, tc, THRESHOLD)
    ok  = "✅" if got == expected else "❌"
    if got != expected:
        all_pass = False
    print(f"{vp:10} {vc:6.2f} {tp:10} {tc:6.2f} {expected:16} {got:16} {ok}")

print()
print("✅  All mismatch cases correct." if all_pass else "❌  Some cases failed — review mismatch logic.")


## 6  Phase 5 Summary

In [ ]:
summary = {
    "Test files":  4,
    "Total test classes": "16+",
    "Config files": 3,
    "Visual model tests": "FaceDetector · CNNEmotionModel · GradCAM · PolarityMapping",
    "Text model tests":   "preprocess_text · TextSentimentModel · get_text_sentiment · AttentionWeights",
    "Fusion tests":       "MismatchDetector · FusionLayer · VisualPolarityBridge",
    "Generator tests":    "SummaryGenerator · build_prompt · edge cases",
    "Mismatch levels":    "MATCH / SOFT_MISMATCH / HARD_MISMATCH",
    "Confidence threshold": THRESHOLD,
}
print("="*60)
print("  Phase 5 — Tests & Config")
print("="*60)
for k, v in summary.items():
    print(f"  {k:<28} {v}")
print("="*60)
print("  Next step: implement models/ and run pytest tests/")
